In [ ]:
import os
os.getcwd()

In [ ]:
import os

print(os.listdir("data"))

In [ ]:
import pandas as pd

train = pd.read_csv("data/training.csv")
test = pd.read_csv("data/test.csv")


In [ ]:

train.head()


In [ ]:
test.head()

In [ ]:
print(train.shape)



In [ ]:
train.info()

In [ ]:
print(test.shape)

In [ ]:
train['SeriousDlqin2yrs'].value_counts()

In [ ]:
#IMPORTS#

In [ ]:
from pathlib import Path
import warnings
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_curve,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

warnings.filterwarnings("ignore")

In [ ]:
RANDOM_STATE = 42
TARGET_COL = "SeriousDlqin2yrs"

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
print("Base directory:", BASE_DIR)

In [ ]:
print("Data directory:", DATA_DIR)

In [ ]:
print("Output directory:", OUTPUT_DIR)

In [ ]:
train_df = pd.read_csv(DATA_DIR / "training.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

In [ ]:
train_df = train_df.drop(columns=["Unnamed: 0"], errors="ignore")
test_df = test_df.drop(columns=["Unnamed: 0"], errors="ignore")

In [ ]:
print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)

In [ ]:
train_df.head()

In [ ]:
print("Training columns:")
print(train_df.columns.tolist())

In [ ]:
print("\nMissing values in training:")
print(train_df.isnull().sum().sort_values(ascending=False))

In [ ]:
print("\nMissing values in test:")
print(test_df.isnull().sum().sort_values(ascending=False))

In [ ]:
print("\nTarget distribution:")
print(train_df[TARGET_COL].value_counts(dropna=False))
print(train_df[TARGET_COL].value_counts(normalize=True).rename("proportion"))

In [ ]:
TARGET_COL = "SeriousDlqin2yrs"

X = train_df.drop(columns=[TARGET_COL], errors="ignore").copy()
y = train_df[TARGET_COL].copy()
X_test = test_df.copy()

common_cols = [c for c in X.columns if c in X_test.columns]
X = X[common_cols].copy()
X_test = X_test[common_cols].copy()

In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)
print("\nFeature columns:")
print(X.columns.tolist())

In [ ]:
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

print("Numeric columns:")
print(numeric_cols)

print("\nCategorical columns:")
print(categorical_cols)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

    preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)

print(preprocessor)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

In [ ]:
pip install xgboost

In [ ]:
from xgboost import XGBClassifier

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    "logistic_regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=20,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    ),
    "gradient_boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

# Optional: XGBoost
try:
    from xgboost import XGBClassifier
    models["xgboost"] = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    )
    print("XGBoost added")
except:
    print("XGBoost not installed")

models

In [ ]:
def compute_ks(y_true, y_score):
    import pandas as pd
    import numpy as np
    
    df = pd.DataFrame({"y": y_true, "score": y_score}).sort_values("score", ascending=False)
    bad = (df["y"] == 1).astype(int)
    good = (df["y"] == 0).astype(int)
    
    cum_bad = bad.cumsum() / bad.sum()
    cum_good = good.cumsum() / good.sum()
    
    ks = np.max(np.abs(cum_bad - cum_good))
    return ks

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
pipelines = {}

for name, model in models.items():
    
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    # Cross-validation
    cv_auc = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc"
    )
    
    # Fit model
    pipeline.fit(X_train, y_train)
    
    # Validation predictions
    prob = pipeline.predict_proba(X_valid)[:, 1]
    
    results.append({
        "model": name,
        "cv_auc": cv_auc.mean(),
        "val_auc": roc_auc_score(y_valid, prob),
        "pr_auc": average_precision_score(y_valid, prob),
        "ks": compute_ks(y_valid, prob)
    })
    
    pipelines[name] = pipeline
    
    print(f"{name} → AUC: {roc_auc_score(y_valid, prob):.4f}, KS: {compute_ks(y_valid, prob):.4f}")

In [ ]:
results_df = pd.DataFrame(results).sort_values(by="val_auc", ascending=False)
results_df


In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_pipeline = pipelines[best_model_name]

print("Best model:", best_model_name)

In [ ]:
best_pipeline

In [ ]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix

prob = best_pipeline.predict_proba(X_valid)[:, 1]
pred = (prob >= 0.5).astype(int)

print("AUC:", roc_auc_score(y_valid, prob))
print("Precision:", precision_score(y_valid, pred))
print("Recall:", recall_score(y_valid, pred))
print("KS:", compute_ks(y_valid, prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_valid, pred))

In [ ]:
results_df = pd.DataFrame(results).sort_values(by="val_auc", ascending=False)
results_df

In [ ]:
test_prob = best_pipeline.predict_proba(X_test)[:, 1]



output = pd.DataFrame({
    "ID": test_df.index,
    "Probability_of_Default": test_prob
})


output.head()

In [ ]:
output_path = OUTPUT_DIR / "test_predictions.csv"

output.to_csv(output_path, index=False)

print("Saved to:", output_path)

In [ ]:
model = best_pipeline.named_steps["model"]

if hasattr(model, "feature_importances_"):
    importances = model.feature_importances_
    print("Top feature importances:")
    print(importances[:10])

In [ ]:
import pandas as pd

model = best_pipeline.named_steps['model']

feature_names = X.columns

importances = model.feature_importances_

feat_imp = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

feat_imp.head(10)

In [ ]:
import matplotlib.pyplot as plt

feat_imp.head(10).plot(
    x="feature",
    y="importance",
    kind="barh",
    figsize=(8,5)
)
plt.gca().invert_yaxis()
plt.title("Top Feature Importances")
plt.show()

In [ ]:
pip install shap

In [ ]:
import shap
print(shap.__version__)

In [ ]:
import shap

explainer = shap.Explainer(best_pipeline.named_steps["model"])
shap_values = explainer(best_pipeline.named_steps["preprocessor"].transform(X_valid))

shap.summary_plot(shap_values, X_valid)

In [ ]:
preprocessor_fitted = best_pipeline.named_steps["preprocessor"]
model_fitted = best_pipeline.named_steps["model"]

print(type(preprocessor_fitted))
print(type(model_fitted))

In [ ]:
X_background = X_train.sample(min(200, len(X_train)), random_state=42)
X_explain = X_valid.head(100).copy()

X_background_t = preprocessor_fitted.transform(X_background)
X_explain_t = preprocessor_fitted.transform(X_explain)

print("Background shape:", X_background_t.shape)
print("Explain shape:", X_explain_t.shape)

In [ ]:
feature_names = X_train.columns.tolist()

print(feature_names)
print("Total features:", len(feature_names))

In [ ]:
explainer = shap.Explainer(
    model_fitted,
    X_background_t,
    feature_names=feature_names
)

shap_values = explainer(X_explain_t)

In [ ]:
print(len(feature_names))
print(X_explain_t.shape)

In [ ]:
shap.summary_plot(shap_values, X_explain_t, feature_names=feature_names)

In [ ]:
param_grid = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [3, 4, 5],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__subsample": [0.7, 0.8],
    "model__colsample_bytree": [0.7, 0.8]
}

In [ ]:
from sklearn.pipeline import Pipeline

xgb_model = XGBClassifier(
    eval_metric="logloss",
    random_state=42
)

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    xgb_pipeline,
    param_grid,
    cv=3,
    scoring="roc_auc",
    verbose=1,
    n_jobs=-1
)

grid.fit(X_train, y_train)

In [ ]:
print("Best parameters:")
print(grid.best_params_)

print("\nBest CV AUC:")
print(grid.best_score_)

In [ ]:
best_xgb_pipeline = grid.best_estimator_

In [ ]:
prob = best_xgb_pipeline.predict_proba(X_valid)[:, 1]

print("AUC:", roc_auc_score(y_valid, prob))
print("KS:", compute_ks(y_valid, prob))